In [1]:
import os
import warnings
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import LeaveOneGroupOut, ParameterGrid
from sklearn.metrics import f1_score, balanced_accuracy_score, recall_score, precision_score, confusion_matrix
from xgboost import XGBClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import catboost as cb
from catboost import CatBoostClassifier
from sklearn.svm import SVR, SVC
from lightgbm import LGBMClassifier
from scikeras.wrappers import KerasClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

In [2]:
def create_sequences_by_participant(df, window_size, feature_cols, target_col):
    X_list, y_list, groups_list = [], [], []
    
    for p_id, group in df.groupby("participant"):
        # Se il paziente ha abbastanza dati per almeno una finestra
        if len(group) >= window_size:
            features = group[feature_cols].values
            target = group[target_col].values
            
            # Crea sequenze solo per questo paziente
            for i in range(len(group) - window_size):
                X_list.append(features[i : i + window_size])
                y_list.append(target[i + window_size])
                groups_list.append(p_id) # Salva il partecipante corretto
                
    return np.array(X_list), np.array(y_list), np.array(groups_list)

In [3]:
def build_lstm_model(n_features, timesteps=1, n_units=64, dropout=0.2, lr=1e-3):

    model = Sequential([
        LSTM(64, input_shape=(timesteps, n_features)),
        Dense(32, activation="relu"),
        Dense(1, activation="sigmoid")
    ])
    
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    #model.summary()
    return model

In [4]:
list(range(0,1))

[0]

In [5]:
# Suppress warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
seed = 69
torch.manual_seed(seed)
np.random.seed(seed)

# Define root directory
root = '.'

In [6]:
df = pd.read_csv('./new_dataset/maison-llf-features.CSV', sep=",")

In [7]:
count_vec = CountVectorizer()

In [8]:
ana = pd.read_csv('./new_dataset/maison-llf-demographics.CSV', sep=",")

In [9]:
ana_col = list(ana.columns)

In [10]:
ana_col

['participant',
 'sex',
 'age',
 'fracture-type',
 'relationship',
 'education',
 'work',
 'ethnicity']

In [11]:
for column in ana_col:

    if column != 'age' and column != 'participant':

        #print(column)
        
        count_matrix = count_vec.fit_transform(ana[column])
        count_df = pd.DataFrame(
            count_matrix.toarray(),
            columns=[f"CountVect_{w}" for w in count_vec.get_feature_names_out()]
        )
        ana = pd.concat([ana, count_df], axis=1)

In [12]:
tfidf_vec = TfidfVectorizer()

In [13]:
for column in ana_col:

    if column != 'age' and column != 'participant':

        #print(column)
        
        count_matrix = tfidf_vec.fit_transform(ana[column])
        count_df = pd.DataFrame(
            count_matrix.toarray(),
            columns=[f"TfidVect_{w}" for w in tfidf_vec.get_feature_names_out()]
        )
        ana = pd.concat([ana, count_df], axis=1)

In [14]:
num_ana = ana.drop(["sex", "fracture-type", "relationship", "education", "work", "ethnicity"], axis=1)

In [15]:
num_ana.head(5)

,participant,age,CountVect_female,CountVect_male,CountVect_femur,CountVect_fracture,CountVect_hip,CountVect_pelvis,CountVect_replacement,CountVect_divorced,...,TfidVect_secondary,TfidVect_undergraduate,TfidVect_employed,TfidVect_part,TfidVect_retired,TfidVect_time,TfidVect_asian,TfidVect_black,TfidVect_caucasian,TfidVect_south
0,1,74,1,0,0,1,1,0,0,0,...,0.000000,0.782749,0.00000,0.00000,1.0,0.00000,0.0,1.0,0.0,0.0
1,2,66,1,0,0,1,1,0,0,0,...,0.000000,0.782749,0.57735,0.57735,0.0,0.57735,0.0,0.0,1.0,0.0
2,3,70,1,0,0,1,0,1,0,0,...,0.707107,0.000000,0.00000,0.00000,1.0,0.00000,0.0,0.0,1.0,0.0
3,4,60,1,0,1,1,0,0,0,1,...,0.707107,0.000000,0.00000,0.00000,1.0,0.00000,0.0,0.0,1.0,0.0
4,5,75,0,1,0,0,1,0,1,0,...,0.000000,0.000000,0.57735,0.57735,0.0,0.57735,0.0,0.0,1.0,0.0


In [16]:
data = df.merge(num_ana, how='right', on='participant')

In [17]:
# Compute quartiles for discretization
siss_q1, siss_q2, siss_q3 = np.percentile(data["sis"], [25, 50, 75])
ohss_q1, ohss_q2, ohss_q3 = np.percentile(data["ohs"], [25, 50, 75])
okss_q1, okss_q2, okss_q3 = np.percentile(data["oks"], [25, 50, 75])

In [18]:
# Define quartile-based bins and labels
quartile_labels = [0, 1, 2, 3]

In [19]:
# Apply discretization
data["SISS_Category_Q"] = pd.cut(data["sis"], bins=[data["sis"].min(), siss_q1, siss_q2, siss_q3, data["sis"].max()],
                               labels=quartile_labels, include_lowest=True).astype(int)
data["OHSS_Category_Q"] = pd.cut(data["ohs"], bins=[data["ohs"].min(), ohss_q1, ohss_q2, ohss_q3, data["ohs"].max()],
                               labels=quartile_labels, include_lowest=True).astype(int)

data["OKSS_Category_Q"] = pd.cut(data["oks"], bins=[data["oks"].min(), okss_q1, okss_q2, okss_q3, data["oks"].max()],
                               labels=quartile_labels, include_lowest=True).astype(int)

In [20]:
# Extract features, target, and patient IDs for LOPO
X = data.iloc[:, -104:-3]
groups = data["participant"]

In [21]:
#84-39 + 1

In [22]:
#X.info()

In [23]:
# Define classifier and hyperparameter grid
param_grid = {
            "model__n_units": [16, 32, 64],
            "model__dropout": [0.0, 0.2],
            "model__lr": [1e-3, 3e-4],
            "batch_size": [16, 32],
            "epochs": [10, 20],
        }

In [24]:
# Leave-One-Patient-Out CV (LOPO)
outer_logo = LeaveOneGroupOut()

In [25]:
# Initialize results storage
overall_conf_matrix = np.zeros((4, 4))  # Assuming 4 categories (0, 1, 2, 3)
performance_metrics = []

In [26]:
# ESEMPIO D'USO:
WINDOW_SIZE = 55 # Consiglio una finestra più piccola (es. una settimana) 
                # per avere più campioni per ogni paziente!  (7)

lista_features = X.columns

X_input, y_input, groups_input = create_sequences_by_participant(
    data, WINDOW_SIZE, lista_features, ['sis', 'ohs', 'oks', 'SISS_Category_Q', 'OHSS_Category_Q', 'OKSS_Category_Q']
)

# Ora le lunghezze saranno TUTTE uguali e coerenti
print(X_input.shape[0], y_input.shape[0], groups_input.shape[0])

18 18 18


In [27]:
data.shape

(1008, 140)

In [28]:
X_input.shape  ### 18 pazienti (righe)

(18, 55, 101)

In [29]:
len(X_input[0][0])

101

In [30]:
y_input  ### 18 pazienti e 6 var risposta

array([[25, 25, 28,  2,  1,  1],
       [18, 22, 45,  0,  0,  3],
       [24, 28, 48,  1,  1,  3],
       [26, 39, 22,  2,  3,  0],
       [29, 46, 44,  3,  3,  3],
       [23, 31, 30,  1,  2,  2],
       [19, 30, 31,  0,  2,  2],
       [22, 20, 24,  1,  0,  1],
       [19, 17, 20,  0,  0,  0],
       [18, 28, 34,  0,  1,  2],
       [20, 46, 48,  0,  3,  3],
       [29, 16, 21,  3,  0,  0],
       [27, 31, 28,  2,  2,  1],
       [27, 29, 23,  2,  2,  1],
       [17, 24, 17,  0,  1,  0],
       [29, 22, 29,  3,  0,  2],
       [27, 26, 18,  2,  1,  0],
       [30, 23, 32,  3,  1,  2]])

In [31]:
y_input[:, 4]

array([1, 0, 1, 3, 3, 2, 2, 0, 0, 1, 3, 0, 2, 2, 1, 0, 1, 1])

In [ ]:
# Outer LOPO Loop
count=0
y_input = y_input[:,2] 

for train_idx, test_idx in outer_logo.split(X_input, y_input, groups_input):
    #print(train_idx) index
    count=count+1
    print(count)
    
    #if count == 6 or count == 7:
    #    continue 
    X_train_outer, X_test = X_input[train_idx], X_input[test_idx] #X_train_outer, X_test = X_input.loc[train_idx].to_numpy(), X.iloc[test_idx].to_numpy()
    y_train_outer, y_test = y_input[train_idx], y_input[test_idx] #y_train_outer, y_test = y_input.iloc[train_idx].to_numpy(), y.iloc[test_idx].to_numpy()
    groups_train_outer = groups_input[train_idx] #groups_train_outer = groups.iloc[train_idx]
    print(np.unique(groups_train_outer)) #print(groups_train_outer.unique())
    
    # Inner LOPO for Hyperparameter Optimization
    inner_logo = LeaveOneGroupOut()
    best_model = None
    best_score = -np.inf
    
    for inner_train_idx, inner_val_idx in inner_logo.split(X_train_outer, y_train_outer, groups_train_outer):
        X_train_inner, X_val = X_train_outer[inner_train_idx], X_train_outer[inner_val_idx]
        y_train_inner, y_val = y_train_outer[inner_train_idx], y_train_outer[inner_val_idx]
        groups_train_inner = groups_train_outer[inner_train_idx] #groups_train_inner = groups_train_outer.iloc[inner_train_idx]
        print(np.unique(groups_train_inner)) #print(groups_train_inner.unique())
        
        # Hyperparameter tuning
        for params in ParameterGrid(param_grid):
            params = {k: int(v) if isinstance(v, np.generic) else v for k, v in params.items()}
            model = KerasClassifier(
                                    model=build_lstm_model,
                                    n_features=len(lista_features),
                                    epochs=10,
                                    batch_size=32,
                                    verbose=0
                                )
            model.set_params(**params)
            model.fit(X_train_inner, y_train_inner)
            y_val_pred = model.predict(X_val)
            score = f1_score(y_val, y_val_pred, average="macro")
            # https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html

            if score > best_score:
                best_score = score
                best_model = model
                best_params = params  # Store the best hyperparameters

    # Train best model on full outer training set
    best_model.fit(X_train_outer, y_train_outer)
    y_pred = best_model.predict(X_test)

    # Compute metrics
    f1 = f1_score(y_test, y_pred, average="macro")
    balanced_acc = balanced_accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred, average="macro")
    precision = precision_score(y_test, y_pred, average="macro")
    conf_matrix = confusion_matrix(y_test, y_pred, labels=[0, 1, 2, 3])

    # Aggregate confusion matrices
    overall_conf_matrix += conf_matrix

    # Store results
    performance_metrics.append([f1, balanced_acc, recall, precision])

1
[ 2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18]
[ 3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18]
[ 2  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18]
[ 2  3  5  6  7  8  9 10 11 12 13 14 15 16 17 18]
[ 2  3  4  6  7  8  9 10 11 12 13 14 15 16 17 18]
[ 2  3  4  5  7  8  9 10 11 12 13 14 15 16 17 18]
[ 2  3  4  5  6  8  9 10 11 12 13 14 15 16 17 18]
[ 2  3  4  5  6  7  9 10 11 12 13 14 15 16 17 18]
[ 2  3  4  5  6  7  8 10 11 12 13 14 15 16 17 18]
[ 2  3  4  5  6  7  8  9 11 12 13 14 15 16 17 18]


In [ ]:
overall_conf_matrix

In [ ]:
# Convert results to DataFrame
performance_df = pd.DataFrame(performance_metrics, columns=["Macro-F1", "Balanced Accuracy", "Macro Recall", "Macro Precision"])

In [ ]:
performance_df

In [ ]:
assert False

In [ ]:
# Save performance metrics and overall confusion matrix
output_path = os.path.join(root, "results/model_performance_ohss.xlsx")
with pd.ExcelWriter(output_path) as writer:
    performance_df.to_excel(writer, sheet_name="All_Folds")

In [ ]:
# Plot overall confusion matrix
plt.figure(figsize=(6, 5))
sns.heatmap(overall_conf_matrix, annot=True, cmap="coolwarm", xticklabels=[0, 1, 2, 3], yticklabels=[0, 1, 2, 3])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Overall Confusion Matrix OHSS")
conf_matrix_path = os.path.join(root, "results/overall_confusion_matrix_ohss.pdf")
plt.savefig(conf_matrix_path, dpi=300, bbox_inches='tight')
plt.close()

print(f"\n✅ Performance metrics saved as: {output_path}")
print(f"✅ Overall Confusion Matrix saved as: {conf_matrix_path}")